# 7. RAG and Agentic Search

Notes for this chapter:

- 

In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5-5"

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, stream=False):
    params = {
        "model": model,
        "max_tokens": 16000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    if stream:
        with client.messages.stream(**params) as response:
            for text in response.text_stream:
                print(text, end="", flush=True)
            message = response.get_final_message()
    else:
        message = client.messages.create(**params)

    # Responses can start with a thinking block, so find the text block
    for block in message.content:
        if block.type == "text":
            return block.text
    raise RuntimeError(f"No text in response (stop_reason={message.stop_reason!r})")

In [3]:
messages = []

add_user_message(messages, "Hello!")
answer = chat(messages)
add_assistant_message(messages, answer)

answer

'Hello! How can I help you today?'

## Exercises

# Retrieval Augmented Generation (RAG)

Retrieval Augmented Generation (RAG) is a technique that helps you work with large documents that are too big to fit into a single prompt. Instead of cramming everything into one massive prompt, RAG breaks documents into chunks and only includes the most relevant pieces when answering questions.

## Large Documents

Imagine you have an 800-page financial document and want to ask Claude specific questions about it, like "What risk factors does this company have?" You need to get the relevant information from the document to Claude somehow, but there are limits to how much text you can include in a prompt.

### Option 1 - Cram everything in the prompt

```
Answer the user's question about the financial document.

<user_question>
{user_question}
</user_question>

<financial_document>
{financial_document}
</financial_document>
```

This approach has serious limitations:

- There's a hard limit on prompt length - your document might be too long
- Claude becomes less effective with very long prompts
- Larger prompts cost more to process
- Larger prompts take longer to process

### Option 2 - Break Documents into Chunks

RAG takes a smarter approach. First, you break the document into smaller chunks during a preprocessing step. 

Then, when a user asks a question, you find the chunks most relevant to their question and only include those in your prompt.

Here's how it works: if someone asks "What risks does this company face?" you'd search through your chunks, find the "Risk Factors" section, and include just that relevant chunk in your prompt.

#### Benefits of RAG

- Claude can focus on only the most relevant content
- Scales up to very large documents
- Works with multiple documents
- Smaller prompts cost less and run faster

#### Challenges with RAG

- Requires a preprocessing step to chunk documents
- Need a search mechanism to find "relevant" chunks
- Included chunks might not contain all the context Claude needs
- Many ways to chunk text - which approach is best?

### When to Use RAG

RAG involves many technical decisions and requires more work than simply including everything in a prompt. You'll need to analyze whether the benefits outweigh the complexity for your particular application. It's especially valuable when working with very large documents, multiple documents, or when you need to optimize for cost and performance.

The key insight is that RAG trades simplicity for scalability and efficiency. While it requires more upfront work to implement properly, it enables you to work with document collections that would be impossible to handle with simple prompt stuffing.


## Text chunking Strategies

### Sized-based chunking

The simplest approach, but has serious limitations:

- Words get cut off mid-sentence
- Chunks lose important context from surrounding text
- Section headers might be separated from their content

An example with some overlapping:

In [ ]:
def chunk_by_char(text, chunk_size=150, chunk_overlap=20):
    chunks = []
    start_idx = 0
    
    while start_idx < len(text):
        end_idx = min(start_idx + chunk_size, len(text))
        chunk_text = text[start_idx:end_idx]
        chunks.append(chunk_text)
        
        start_idx = (
            end_idx - chunk_overlap if end_idx < len(text) else len(text)
        )
    
    return chunks

### Structure-Based Chunking

Use the documents natural structure to split

For example in HTML, you could splits h1s, h2s, body, paragraph...

In a text document you could split by new line breaks

In [6]:
import re


def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

### Semantic-Based Chunking

Semantic-based chunking is the most sophisticated approach. You divide text into sentences, then use natural language processing to determine how related consecutive sentences are. You build chunks from groups of related sentences.

This method is computationally expensive but produces the most relevant chunks. It requires understanding the meaning of individual sentences and is more complex to implement than the other strategies.

### Sentence-Based Chunking

A practical middle ground is chunking by sentences. You split the text into individual sentences using regular expressions, then group them into chunks with optional overlap:



In [ ]:
def chunk_by_sentence(text, max_sentences_per_chunk=5, overlap_sentences=1):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    
    chunks = []
    start_idx = 0
    
    while start_idx < len(sentences):
        end_idx = min(start_idx + max_sentences_per_chunk, len(sentences))
        current_chunk = sentences[start_idx:end_idx]
        chunks.append(" ".join(current_chunk))
        
        start_idx += max_sentences_per_chunk - overlap_sentences
        
        if start_idx < 0:
            start_idx = 0
    
    return chunks

### Which should we choose?

- Structure-based: Best results when you control document formatting (like internal company reports)
- Sentence-based: Good middle ground for most text documents
- Size-based: Most reliable fallback that works with any content type, including code

Size-based chunking with overlap is often the go-to choice in production because it's simple, reliable, and works with any document type. While it may not give perfect results, it consistently produces reasonable chunks that won't break your pipeline.

Remember: there's no single "best" chunking strategy. The right approach depends on your specific documents, use cases, and the trade-offs you're willing to make between implementation complexity and chunk quality.

## Text Embeddings

After breaking a document into chunks, the next step in a RAG pipeline is finding which chunks are most relevant to a user's question. 

This is essentially a search problem - you need to look through all your text chunks and identify the ones that relate to what the user is asking about.

### Semantic Search

The most common approach for finding relevant chunks is semantic search. Unlike keyword-based search that looks for exact word matches, semantic search uses text embeddings to understand the meaning and context of both the user's question and each text chunk.

### Text Embeddings

A text embedding is a numerical representation of the meaning contained in some text. Think of it as converting words and sentences into a format that computers can work with mathematically.

Here's how the process works:

1. You feed text into an embedding model
2. The model outputs a long list of numbers (the embedding)
3. Each number ranges from -1 to +1
4. These numbers represent different qualities or features of the input text

Each number in an embedding is essentially a "score" for some quality of the input text. However, here's the important caveat: we don't know precisely what each number represents.


For example 

```
[0.4, -1, 0.9, 0]

Might corresponsend to

[Positivity, How much the text talks about fruit, How much the text talks about oceans, How formal the text is]

```

## VoyageAI

The recommended approach for embadding is [VoyageAI](https://www.voyageai.com)

In [25]:
from dotenv import load_dotenv
from voyageai.client import Client as VoyageClient

load_dotenv()
client = VoyageClient()

def generate_embeddings(chunks, model="voyage-3-large", input_type="query"):
    result = client.embed(chunks, model=model, input_type=input_type)
    return result.embeddings

## Full RAG pipleine 

1. Chunk your source text
2. Generate your embeddings. This usually handles normalisation for you, so that all values are -1 <= x <= 1
3. Store in your vector database. This is a specialised database for storing, reading and handling long lists of numbers.
4. Process user input. Run the query through the same embedding model.
5. Find similar embeddings. Cosine similarity is used to close matches in our embeddings.


E.g. 

A sentence about medical science might give [0.9, -0.5] where the array is [Medical Topics, Software bugs]
A sentence about software engineering might give [-0.4, 0.6]

A user query about software engineering in the report might be [-0.2, 0.5]. Using cosine similarity, this is closer to the software engineering embedding.


The formula is 

`A x B / ||A|| x ||B||`

Cosine distance is also often used as a metric to display similarity. This is simply `1 - cosine similarity`

6. Create the final prompt

Using the cosine similarity, select the sentence to include with the user query:

```
Answer the user's question about the financial document.

<user_question>
How many bugs did engineers fix this year?
</user_question>

<report>
## Section 2: Software Engineering
This division dedicated significant effort to studying various infection vectors in our distributed systems
</report>

```


## Why is RAG better?

RAG still wins in most cases because the embedding work is cheap, happens once, and is reused.

1. You embed once and query many times. The documents are embedded ahead of time, when they’re indexed, and the vectors are stored. After that, each question costs only:

embedding the question (a few dozen tokens),
a vector search, which takes milliseconds even across millions of chunks,
an LLM call over perhaps 2–5K tokens of retrieved text.

Without RAG, every question pays to push the whole document through the LLM again. With 1,000 questions, RAG processes the document about once, while long-context prompting processes it 1,000 times.

2. Embedding models are much cheaper than LLMs. An embedding model is small (often 0.1–8B parameters, against far larger chat models). It makes one forward pass and outputs a single vector, with no token-by-token generation. API prices show the gap: about $0.02–0.15 per million tokens to embed, against several dollars per million input tokens for a capable LLM.

3. Long prompts get disproportionately expensive. Attention cost grows faster than linearly with context length. A 200K-token prompt costs far more compute and latency than 50 prompts of 4K tokens. Time-to-first-token on a very long prompt can run to tens of seconds.

4. Sometimes the content can’t fit at all. A corpus of thousands of documents, such as years of reports or a literature collection, can exceed any context window. RAG is the only way to search it.

5. Quality is often better too. LLMs tend to miss details buried in very long contexts (the “lost in the middle” effect). A focused set of relevant chunks often gives sharper answers.

When RAG isn’t faster

For one document and a handful of questions, putting the whole thing in the prompt is often simpler and just as cheap. Modern models handle 200K+ tokens, and prompt caching makes repeated questions on the same document much cheaper. RAG pays off as the number of documents, the number of queries, or both grow.
   


In [26]:
import numpy as np
from anthropic_building_with_claude.vector_index import VectorIndex

with open("./report.md", "r") as f:
    text = f.read()

chunks = chunk_by_section(text)

embeddings = generate_embeddings(chunks)


## Store this in a vector database for quick access

store = VectorIndex()

for embedding, chunk in zip(embeddings, chunks):
    store.add_vector(embedding, {"content": chunk})

def cosine_similarity(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))



In [27]:

user_query = "How many oceans are there in the world, according to that report?"

user_query_embedding = generate_embeddings([user_query])[0]


In [ ]:
user_query = "Did people really go to the moon?"

user_query_embedding = generate_embeddings([user_query])[0]

In [32]:
best_embedding = None

results = store.search(user_query_embedding, 2)

top_result = results[0]

if not top_result:
    raise RuntimeError("Expected Top Result")

prompt = f"""
Answer the user's question about the report.

<user_question>
{user_query}
</user_question>

<report>
{top_result[0]["content"][0:200]}
</report>

"""

print(prompt)


Answer the user's question about the report.

<user_question>
How many oceans are there in the world, according to that report?
</user_question>

<report>
## My Report

This is all of the interesting things that I've learn about oceans: 

- They contain a lot of water
- They have some fish in 
- They're kind of big
- There's more than 1 ocean in the wor
</report>


